In [1]:
import numpy as np
import pandas as pd
import os
import json
from matplotlib import pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
def load_splits_csv(out_dir: str):
    """
    Загружает train / val / test + metadata.json
    """
    df_train = pd.read_csv(f"{out_dir}/train.csv")
    df_val   = pd.read_csv(f"{out_dir}/val.csv")
    df_test  = pd.read_csv(f"{out_dir}/test.csv")

    meta = None
    meta_path = f"{out_dir}/metadata.json"
    if os.path.exists(meta_path):
        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)

    return {
        "df_train": df_train,
        "df_val": df_val,
        "df_test": df_test,
        "metadata": meta,
    }

In [3]:
spl = load_splits_csv("/content/drive/MyDrive/monkeytype/splits_v1")
df_train = spl["df_train"]
df_val   = spl["df_val"]
df_test  = spl["df_test"]

In [4]:
for col in df_train.columns:
    print(col)

x1_linspace
x1_brightness
x1_time_reaction
x1_symbol_m
x1_symbol_c
x1_symbol_s
x1_symbol_y
x1_symbol_f
x1_symbol_j
x1_location_0
x1_location_1
x1_location_2
x1_location_3
x1_participant_J
x1_participant_F
x1_participant_Y
x1_correct_symbol_m
x1_correct_symbol_c
x1_correct_symbol_s
x1_correct_symbol_y
x1_correct_symbol_f
x1_correct_symbol_j
x2_linspace
x2_brightness
x2_time_reaction
x2_symbol_m
x2_symbol_c
x2_symbol_s
x2_symbol_y
x2_symbol_f
x2_symbol_j
x2_location_0
x2_location_1
x2_location_2
x2_location_3
x2_participant_J
x2_participant_F
x2_participant_Y
x2_correct_symbol_m
x2_correct_symbol_c
x2_correct_symbol_s
x2_correct_symbol_y
x2_correct_symbol_f
x2_correct_symbol_j
x3_linspace
x3_brightness
x3_time_reaction
x3_symbol_m
x3_symbol_c
x3_symbol_s
x3_symbol_y
x3_symbol_f
x3_symbol_j
x3_location_0
x3_location_1
x3_location_2
x3_location_3
x3_participant_J
x3_participant_F
x3_participant_Y
x3_correct_symbol_m
x3_correct_symbol_c
x3_correct_symbol_s
x3_correct_symbol_y
x3_correct_sym

In [5]:
df_train['location'] = (
    df_train.filter(like='x1_location_')
      .idxmax(axis=1)
      .str.split('_')
      .str[-1]
      .astype(int)
)

df_train['location']

,location
0,2
1,3
2,3
3,2
4,0
...,...
2800,1
2801,3
2802,2
2803,1


In [6]:
p_cols = [c for c in df_train.columns if c.startswith("x1_participant_")]

df_train["participant"] = (
    df_train[p_cols]
      .idxmax(axis=1)
      .str.replace("x1_participant_", "", regex=False)
)
df_train["participant"]

,participant
0,Y
1,F
2,F
3,Y
4,J
...,...
2800,J
2801,F
2802,F
2803,J


In [7]:
result = (
    df_train.groupby(["participant", "x1_linspace_session_id"])["location"]
      .agg(lambda s: s.value_counts().idxmax())   # мода
      .reset_index(name="most_popular_location")
)

result

,participant,x1_linspace_session_id,most_popular_location
0,F,0.000000,1
1,F,0.035714,1
2,F,0.071429,1
3,F,0.107143,2
4,F,0.142857,1
...,...,...,...
101,Y,0.878788,2
102,Y,0.909091,2
103,Y,0.939394,2
104,Y,0.969697,2


In [8]:
df_train = df_train.merge(
    result,
    on=["participant", "x1_linspace_session_id"],
    how="left"
)

In [9]:
df_train["most_popular_location"]

,most_popular_location
0,2
1,3
2,3
3,2
4,1
...,...
2800,1
2801,1
2802,1
2803,1


In [10]:
p_cols = [c for c in df_test.columns if c.startswith("x1_participant_")]

df_test["participant"] = (
    df_test[p_cols]
      .idxmax(axis=1)
      .str.replace("x1_participant_", "", regex=False)
)
df_test["participant"]

,participant
0,F
1,J
2,J
3,F
4,J
...,...
930,Y
931,Y
932,F
933,Y


In [11]:
df_test = df_test.merge(
    df_train[
        ["participant", "x1_linspace_session_id", "most_popular_location"]
    ].drop_duplicates(),
    on=["participant", "x1_linspace_session_id"],
    how="left"
)

In [12]:
p_cols = [c for c in df_val.columns if c.startswith("x1_participant_")]

df_val["participant"] = (
    df_val[p_cols]
      .idxmax(axis=1)
      .str.replace("x1_participant_", "", regex=False)
)
df_val["participant"]

,participant
0,F
1,J
2,J
3,J
4,F
...,...
930,F
931,Y
932,J
933,Y


In [13]:
df_val = df_val.merge(
    df_train[
        ["participant", "x1_linspace_session_id", "most_popular_location"]
    ].drop_duplicates(),
    on=["participant", "x1_linspace_session_id"],
    how="left"
)

In [14]:
df_test['location'] = (
    df_test.filter(like='x1_location_')
      .idxmax(axis=1)
      .str.split('_')
      .str[-1]
      .astype(int)
)

df_test['location']

,location
0,1
1,1
2,0
3,2
4,0
...,...
930,0
931,2
932,2
933,2


In [15]:
df_val['location'] = (
    df_val.filter(like='x1_location_')
      .idxmax(axis=1)
      .str.split('_')
      .str[-1]
      .astype(int)
)

df_val['location']

,location
0,2
1,2
2,2
3,2
4,2
...,...
930,1
931,2
932,2
933,2


In [16]:
y_pred = df_test["most_popular_location"]
y_test = df_test['location']

In [17]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy of baseline classifier:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy of baseline classifier: 0.46737967914438505
              precision    recall  f1-score   support

           0       0.25      0.01      0.02       106
           1       0.49      0.68      0.57       419
           2       0.45      0.38      0.41       329
           3       0.36      0.31      0.33        81

    accuracy                           0.47       935
   macro avg       0.39      0.35      0.33       935
weighted avg       0.44      0.47      0.43       935



In [18]:
y_pred_val = df_val["most_popular_location"]
y_val = df_val['location']

In [19]:
print("Accuracy of baseline classifier:", accuracy_score(y_val, y_pred_val))
print(classification_report(y_val, y_pred_val))

Accuracy of baseline classifier: 0.4919786096256685
              precision    recall  f1-score   support

           0       0.11      0.01      0.02       106
           1       0.52      0.69      0.59       424
           2       0.49      0.43      0.46       329
           3       0.35      0.29      0.32        76

    accuracy                           0.49       935
   macro avg       0.37      0.36      0.35       935
weighted avg       0.45      0.49      0.46       935



In [20]:
y_pred_train = df_train["most_popular_location"]
y_train = df_train['location']

In [21]:
print("Accuracy of baseline classifier:", accuracy_score(y_train, y_pred_train))
print(classification_report(y_train, y_pred_train))

Accuracy of baseline classifier: 0.5336898395721925
              precision    recall  f1-score   support

           0       0.38      0.03      0.06       278
           1       0.54      0.71      0.62      1261
           2       0.54      0.50      0.52       999
           3       0.45      0.33      0.38       267

    accuracy                           0.53      2805
   macro avg       0.48      0.39      0.39      2805
weighted avg       0.52      0.53      0.50      2805



In [20]:
df_P = df_test[df_test["participant"] == "Y"]
y_pred = df_P["most_popular_location"]
y_test = df_P['location']
print("Accuracy of baseline classifier:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy of baseline classifier: 0.4819672131147541
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        13
           1       0.44      0.44      0.44       133
           2       0.51      0.58      0.54       153
           3       0.00      0.00      0.00         6

    accuracy                           0.48       305
   macro avg       0.24      0.25      0.25       305
weighted avg       0.45      0.48      0.46       305



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [21]:
#mid бейзлайн
result = (
    df_train.groupby(["participant"])["location"]
      .agg(lambda s: s.value_counts().idxmax())   # мода
      .reset_index(name="most_popular_location2")
)

result

,participant,most_popular_location2
0,F,1
1,J,1
2,Y,2


In [22]:
df_train = df_train.merge(
    result,
    on=["participant"],
    how="left"
)

In [23]:
df_train["most_popular_location2"]

,most_popular_location2
0,2
1,1
2,1
3,2
4,1
...,...
2800,1
2801,1
2802,1
2803,1


In [24]:
df_test = df_test.merge(
    df_train[
        ["participant", "most_popular_location2"]
    ].drop_duplicates(),
    on=["participant",],
    how="left"
)

In [25]:
df_val = df_val.merge(
    df_train[
        ["participant", "most_popular_location2"]
    ].drop_duplicates(),
    on=["participant",],
    how="left"
)

In [30]:
df_P = df_test[df_test["participant"] == "Y"]
y_pred = df_P["most_popular_location2"]
y_test = df_P['location']
print("Accuracy of baseline classifier:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy of baseline classifier: 0.5016393442622951
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        13
           1       0.00      0.00      0.00       133
           2       0.50      1.00      0.67       153
           3       0.00      0.00      0.00         6

    accuracy                           0.50       305
   macro avg       0.13      0.25      0.17       305
weighted avg       0.25      0.50      0.34       305



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [31]:
#weak бейзлайн
result = (
    df_train["location"]
      .agg(lambda s: s.value_counts().idxmax())   # мода
      #.reset_index(name="most_popular_location3")
)

result

np.int64(1)

In [32]:
df_train['most_popular_location3'] = result

In [33]:
df_test = df_test.merge(
    df_train[
        ["participant", "most_popular_location3"]
    ].drop_duplicates(),
    on=["participant",],
    how="left"
)

In [34]:
df_val = df_val.merge(
    df_train[
        ["participant", "most_popular_location3"]
    ].drop_duplicates(),
    on=["participant",],
    how="left"
)

In [40]:
df_P = df_test[df_test["participant"] == "Y"]
y_pred = df_P["most_popular_location3"]
y_test = df_P['location']
print("Accuracy of baseline classifier:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy of baseline classifier: 0.4360655737704918
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        13
           1       0.44      1.00      0.61       133
           2       0.00      0.00      0.00       153
           3       0.00      0.00      0.00         6

    accuracy                           0.44       305
   macro avg       0.11      0.25      0.15       305
weighted avg       0.19      0.44      0.26       305



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
